In [2]:
import numpy as np 
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt                               
import os 
import gc
import pickle
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1 import make_axes_locatable
matplotlib.rcParams["figure.dpi"] = 150

In [3]:
datadir = '/home/mwells5/Muon_Collider_Smart_Pixels/Data_Files/Data_Set_2026Feb_copy_m/Parquet_Files/'
flp = 0

In [ ]:
labels_c = pd.DataFrame()
recon3D = pd.DataFrame()

recon3D_list = []
labels_list = []

count=0
for file in os.listdir(datadir):
    if "recon3D" in file:
        if "bib" in file: 
            recon3D_list.append(pd.read_parquet(f"{datadir}{file}"))
            file = file.replace("recon3D","labels")
            labels_list.append(pd.read_parquet(f"{datadir}{file}"))
            count+=1
        if count == 100:
            break
            
recon3D = pd.concat(recon3D_list)
labels_c = pd.concat(labels_list)

del recon3D_list
del labels_list
gc.collect()

print("Total number of clusters: ", labels_c.shape[0])

Total number of clusters:  47318


In [12]:

pklPath_labels = "/home/dabadjiev/smartpixels_ml_dsabadjiev/Muon_Collider_Smart_Pixels/Data_Files/Data_Set_2026Feb/plots/dfOfTruth.pkl"
pklPath_recon = "/home/dabadjiev/smartpixels_ml_dsabadjiev/Muon_Collider_Smart_Pixels/Data_Files/Data_Set_2026Feb/plots/dfOfRecon.pkl"

labels = pd.DataFrame()
recon2D = pd.DataFrame()
#recon3D = pd.DataFrame()

with open(pklPath_labels, 'rb') as file:
    # Reconstruct the Python object
    truthbib = pickle.load(file)

with open(pklPath_recon, 'rb') as file:
    # Reconstruct the Python object
    recon2Dbib = pickle.load(file)

# parquetData = parquetData[parquetData['source']=='bib_mm']
# parquetData = parquetData.iloc[0:46031]
print(f"Total # of clusters: {len(labels)}")
print(f"keys parquets {labels.keys()} ")


Total # of clusters: 0
keys parquets RangeIndex(start=0, stop=0, step=1) 


In [24]:
def cut_data(data_df, recon_df):
    recon_df = recon_df[(data_df['z-global'] >= 0) & (data_df['z-global'] <= 13)]
    data_df = data_df[(data_df['z-global'] >= 0) & (data_df['z-global'] <= 13)]

    # recon_df = recon_df[(data_df['adjusted_hit_time'] >= -.09) & (data_df['adjusted_hit_time'] <= .15)]
    # data_df = data_df[(data_df['adjusted_hit_time'] >= -.09) & (data_df['adjusted_hit_time'] <= .15)]

    # recon_df = data_df[(data_df['source'] != 'sig')]
    # data_df = data_df[(data_df['source'] != 'sig')]
    
    # recon_df = recon_df[data_df['moduleID'] == 1]
    # data_df = data_df[data_df['moduleID'] == 1]
    
    recon_df.reset_index()
    data_df.reset_index()

    print(f"Total # of clusters (after cuts): {data_df.shape[0]}")
    print(f"keys parquets {data_df.keys()} ")
    return data_df, recon_df

cut_data(labels_c, recon3D)
print(labels_c['hit_time'].min())
neg_hits = labels_c[labels_c['hit_time']<0]
print("number of negative hits: ", neg_hits.shape[0])
print(neg_hits)

Total # of clusters (after cuts): 3656
keys parquets Index(['x-entry', 'y-entry', 'z-entry', 'n_x', 'n_y', 'n_z', 'number_eh_pairs',
       'y-local', 'z-global', 'pt', 'hit_time', 'PID', 'cotAlpha', 'cotBeta',
       'y-midplane', 'x-midplane', 'adjusted_hit_time',
       'adjusted_hit_time_30ps_gaussian', 'adjusted_hit_time_60ps_gaussian'],
      dtype='object') 
-0.2097
number of negative hits:  2966
       x-entry    y-entry  z-entry       n_x       n_y       n_z  \
0    61.402950 -44.997768      0.0 -0.096851  0.096968  0.046610   
6    -8.925150  19.190516      0.0  0.084180 -0.004619  0.107436   
7    20.310762 -38.805817      0.0 -0.099468  0.011818  0.080285   
8   -34.496017  27.709215      0.0  0.001414  0.005895  0.020993   
9   -24.265512 -94.747162      0.0  0.006322  0.052950  0.011847   
..         ...        ...      ...       ...       ...       ...   
146  11.806474 -20.387602      0.0 -0.036940  0.029539  0.122274   
213  98.351944 -59.702824      0.0 -0.073383  0.0

In [ ]:
def populate_module_time(data_df, recon_df):

    data_df, recon_df = cut_data(data_df, recon_df)
    
    # create cluster frames
    clusters = recon_df.to_numpy().reshape(recon_df.shape[0],20,13,21)

    x_l = (data_df['z-global']%13)*40 # changes range from 0 to 13 and converts to px
    x_l = x_l.to_numpy()
    x_l = x_l.astype(int)

    y_l = ((data_df['y-local'])+8.5)*40 # changes range from 0 to 21 and convert to px
    y_l = y_l.to_numpy()
    y_l = y_l.astype(int)

    h_t = (data_df['hit_time']*1e3) # gets impact times for each hit IN PICOSECONDS
    h_t = h_t.to_numpy()
    print(h_t.min(), h_t.max())

    t_slice = 200 # picoseconds. work on this for different t_slices??
    t_start = np.floor(h_t.min() / t_slice) * t_slice
    t_end = h_t.max() + 19*200
    print(t_start, t_end, (t_end-t_start))

    # gets number of frames in the animation for specified start, end, dt
    frame_number = np.ceil((t_end-t_start)/t_slice)

    limit = 60
    if frame_number > limit:
         frame_number = limit
         print(f"Frame limit reached. Only using the first {limit} frames.")

    mod_array_list = np.zeros((frame_number, 520, 520))

    # saves timestamps for each frame to the nearest picosecond
    timestamps = []
    
    j=0
    for j in range(frame_number):
         if ((j*t_slice)+t_start) < 1e3:
             timestamps.append(f"{int(np.floor(np.abs((j*t_slice)+t_start)))}ps")
         else:
             timestamps.append(f"{int(np.ceil((j*t_slice)+t_start)/1e3)}ns")

    # adds hits from 3D clusters to groups of frames
    i=0
    for i in range(recon_df.shape[0]):
      # defines time start for a hit
      t_min = h_t[i]

      if t_min < 0:
           start_frame_id = int(np.ceil(np.abs((t_start-t_min)/t_slice)))
      else:
           start_frame_id = int(np.ceil((t_start+t_min)/t_slice))

      end_frame_id = start_frame_id+int(np.ceil(20*t_slice/200))

      if (start_frame_id>frame_number):
           continue
      elif (end_frame_id>frame_number):
           end_frame_id = frame_number
           clusters[i, :(end_frame_id+1)-(start_frame_id+1)]

      cluster_end = 20-(end_frame_id-start_frame_id) 
      print(start_frame_id, end_frame_id, (end_frame_id-start_frame_id))

      # defines x and y range on module
      cx_min = x_l[i] - 6
      cy_min = y_l[i] - 10
      cx_max = x_l[i] + 7
      cy_max = y_l[i] + 11
    
      # defines x and y range in cluster window
      wx_min = 0
      wy_min = 0 
      wx_max = 13
      wy_max = 21
    
      if cx_min < 0:
           wx_min -= cx_min
           cx_min = 0
      if cy_min < 0:
           wy_min -= cy_min
           cy_min = 0
      if cx_max > 520:
           wx_max -= (cx_max-520)
           cx_max = 520
      if cy_max > 520:
           wy_max -= (cy_max-520)
           cy_max = 520

      mod_array_list[start_frame_id:end_frame_id, cx_min:cx_max, cy_min:cy_max] += np.abs(clusters[i, cluster_end:, wx_min:wx_max, wy_min:wy_max])

      if (frame_number>end_frame_id):
           extras = np.full_like(mod_array_list[end_frame_id:, cx_min:cx_max, cy_min:cy_max], np.abs(clusters[i, 19, wx_min:wx_max, wy_min:wy_max]))
           mod_array_list[end_frame_id:, cx_min:cx_max, cy_min:cy_max] += extras 
           
    eh_max = np.max(mod_array_list)
    print(mod_array_list.shape[0])
    
    return mod_array_list, timestamps, eh_max

In [32]:

def plotMod(mod_array, time, datamax):
    fig, ax = plt.subplots(figsize=(10,10),dpi=150)

    datamin = 0.9

    im = ax.imshow(mod_array,  
                   cmap='magma_r', 
                   interpolation='nearest', norm=mcolors.LogNorm(vmin=datamin, vmax=datamax))
    
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='4%', pad=0.05)
    fig.colorbar(im, cax=cax, location='right',label='Number of eh pairs')
    ax.set_title("Charge for a z-slice, first 100 parquets")

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='4%', pad=0.05)
    fig.colorbar(im, cax=cax, location='right',label='Number of eh pairs')

    # Draw grid on both
    ax.set_xlim(0,520)
    ax.set_ylim(0,520)
    ax.set_xlabel("x-local [px]")
    ax.set_ylabel("y-local [px]")
    plt.figtext(0.12,0.15, f"Elapsed time: {time}", fontsize=20, 
                bbox=dict(facecolor='white', alpha=0.9))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(40))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(40))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(20))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(20))
    plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    plt.tick_params(axis='y', which='both', left=False, right=False, labelleft=False)

    plt.tight_layout(pad=3.5)
    filename = f'frames_holder/frame_{time}.png'
    plt.savefig(f"/home/mwells5/Muon_Collider_Smart_Pixels/megan/tests/{filename}")
    # fig.canvas.draw()
    return filename


In [34]:
from PIL import Image

def make_gif(data_df, recon_df):
    k=0
    frame_files = []
    arrays, times, dmax = populate_module_time(data_df, recon_df)
    for k in range(len(times)-1):
        frame_files.append(plotMod(arrays[k], times[k], dmax))

    images = [Image.open(file) for file in frame_files]

    images[0].save(
    'module_animation4.gif',
    append_images=images[1:], 
    duration=500, # duration for each frame in milliseconds 
    loop=0) # 0 means loop infinitely
    #Image(url='modue_animation.gif')

make_gif(labels_c, recon3D)

Total # of clusters (after cuts): 3656
keys parquets Index(['x-entry', 'y-entry', 'z-entry', 'n_x', 'n_y', 'n_z', 'number_eh_pairs',
       'y-local', 'z-global', 'pt', 'hit_time', 'PID', 'cotAlpha', 'cotBeta',
       'y-midplane', 'x-midplane', 'adjusted_hit_time',
       'adjusted_hit_time_30ps_gaussian', 'adjusted_hit_time_60ps_gaussian'],
      dtype='object') 
-7.92 1089579.8339999998


UnboundLocalError: local variable 't_slice' referenced before assignment